In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [2]:
import warnings
warnings.simplefilter('ignore')

In [3]:
from fastai.text.all import *
from datasets import load_dataset, Dataset, DatasetDict

In [4]:
def prepare_snli_as_text_files(base_path='snli_data', sample_frac=0.1):
    ds = load_dataset("snli")
    label_names = {0: 'entailment', 1: 'neutral', 2: 'contradiction'}
    splits = ['train', 'validation', 'test']
    for split in splits:
        print(f"Processing {split} split...")
        split_ds = ds[split]
        n_samples = int(len(split_ds) * sample_frac)
        snli = split_ds.select(range(n_samples))
        df = pd.DataFrame(snli)
        df = df[df['label'] != -1]
        for label_num, label_name in label_names.items():
            label_examples = df[df['label'] == label_num]
            split_dir = os.path.join(base_path, split, label_name)
            os.makedirs(split_dir, exist_ok=True)
            for idx, row in label_examples.iterrows():
                text = f"Premise: {row['premise']}\n\nHypothesis: {row['hypothesis']}\n\nLabel: {label_name}"
                file_path = os.path.join(split_dir, f"example_{idx}.txt")
                with open(file_path, 'w', encoding='utf-8') as f:
                    f.write(text)

    print(f"Dataset prepared at: {base_path}")
    return base_path

In [5]:
path = prepare_snli_as_text_files('snli_text_files')

Processing train split...
Processing validation split...
Processing test split...
Dataset prepared at: snli_text_files


In [6]:
files = get_text_files(path, folders=['train', 'validation', 'test'])

In [7]:
txt = files[0].open().read(); txt[:75]

'Premise: couples enjoying a meal, while a bunch of dogs take a nap.\n\nHypoth'

In [8]:
spacy = WordTokenizer()
toks = first(spacy([txt]))
print(coll_repr(toks, 30))

['Premise', ':', 'couples', 'enjoying', 'a', 'meal', ',', 'while', 'a', 'bunch', 'of', 'dogs', 'take', 'a', 'nap', '.', '\n\n', 'Hypothesis', ':', 'SOME', 'ANIMALS', 'ARE', 'ASLEEP', '\n\n', 'Label', ':', 'entailment']


In [9]:
tkn = Tokenizer(spacy)
print(coll_repr(tkn(txt), 31))

(#35) ['xxbos', 'xxmaj', 'premise', ':', 'couples', 'enjoying', 'a', 'meal', ',', 'while', 'a', 'bunch', 'of', 'dogs', 'take', 'a', 'nap', '.', '\n\n', 'xxmaj', 'hypothesis', ':', 'xxup', 'some', 'xxup', 'animals', 'xxup', 'are', 'xxup', 'asleep', '\n\n'...]


In [10]:
txts  =L(o.open().read() for o in files[:2000])

In [11]:
def subword(sz):
    sp = SubwordTokenizer(vocab_sz=sz)
    sp.setup(txts)
    return ' '.join(first(sp([txt]))[:40])

In [12]:
subword(1000)

sentencepiece_trainer.cc(178) LOG(INFO) Running command: --input=tmp/texts.out --vocab_size=1000 --model_prefix=tmp/spm --character_coverage=0.99999 --model_type=unigram --unk_id=9 --pad_id=-1 --bos_id=-1 --eos_id=-1 --minloglevel=2 --user_defined_symbols=▁xxunk,▁xxpad,▁xxbos,▁xxeos,▁xxfld,▁xxrep,▁xxwrep,▁xxup,▁xxmaj --hard_vocab_limit=false


'▁P rem ise : ▁couple s ▁enjoy ing ▁a ▁me al , ▁whil e ▁a ▁b unch ▁of ▁dogs ▁take ▁a ▁nap . ▁H yp othes is : ▁S O M E ▁A N I M A L S ▁A'

In [13]:
toks = tkn(txt)
print(coll_repr(tkn(txt), 31))

(#35) ['xxbos', 'xxmaj', 'premise', ':', 'couples', 'enjoying', 'a', 'meal', ',', 'while', 'a', 'bunch', 'of', 'dogs', 'take', 'a', 'nap', '.', '\n\n', 'xxmaj', 'hypothesis', ':', 'xxup', 'some', 'xxup', 'animals', 'xxup', 'are', 'xxup', 'asleep', '\n\n'...]


In [14]:
toks200 = txts[:200].map(tkn)
toks200[0]

['xxbos', 'xxmaj', 'premise', ':', 'couples', 'enjoying', 'a', 'meal', ',', 'while', 'a', 'bunch', 'of', 'dogs', 'take', 'a', 'nap', '.', '\n\n', 'xxmaj', 'hypothesis', ':', 'xxup', 'some', 'xxup', 'animals', 'xxup', 'are', 'xxup', 'asleep', '\n\n', 'xxmaj', 'label', ':', 'entailment']

In [15]:
num = Numericalize()
num.setup(toks200)
coll_repr(num.vocab, 20)

"(#264) ['xxunk', 'xxpad', 'xxbos', 'xxeos', 'xxfld', 'xxrep', 'xxwrep', 'xxup', 'xxmaj', ':', 'a', '\\n\\n', '.', 'premise', 'hypothesis', 'label', 'entailment', 'the', 'in', 'is'...]"

In [16]:
nums = num(toks)[:20]; nums

TensorText([ 2,  8, 13,  9,  0,  0, 10,  0, 29, 44, 10,  0, 22, 81,  0, 10,  0,
            12, 11,  8])

In [17]:
' '.join(num.vocab[o] for o in nums)

'xxbos xxmaj premise : xxunk xxunk a xxunk , while a xxunk of dogs xxunk a xxunk . \n\n xxmaj'

In [18]:
nums200 = toks200.map(num)

In [19]:
dl = LMDataLoader(nums200)

In [20]:
x, y = first(dl)
x.shape, y.shape

(torch.Size([64, 72]), torch.Size([64, 72]))

In [21]:
' '.join(num.vocab[o] for o in y[0][:20])

'xxmaj premise : xxunk xxunk a xxunk , while a xxunk of dogs xxunk a xxunk . \n\n xxmaj hypothesis'

In [22]:
get_ds = partial(get_text_files, folders=['train', 'validation', 'test'])

dls_lm = DataBlock(
    blocks=TextBlock.from_folder(path, is_lm=True), 
    get_items = get_ds, splitter=RandomSplitter(0.1),
).dataloaders(path, path=path, bs=128, seq_len=80)

In [23]:
dls_lm.show_batch(max_n=2)

xxbos xxmaj premise : xxmaj two old men in hats doze in the sun outside . 

 xxmaj hypothesis : xxmaj the two men are fishing . 

 xxmaj label : contradiction xxbos xxmaj premise : a man wearing a black coat and red hat holds a camera . 

 xxmaj hypothesis : a man has a camera . 

 xxmaj label : entailment xxbos xxmaj premise : a woman stands in front of a bridge , talking passionately through
on the railroad . 

 xxmaj hypothesis : a railroad is being worked on 

 xxmaj label : entailment xxbos xxmaj premise : xxmaj in a martial arts demonstration a man with two swords faces a woman with a xxmaj chinese spear . 

 xxmaj hypothesis : xxmaj the kids are playing . 

 xxmaj label : contradiction xxbos xxmaj premise : a crowd is gathered near a riverbank . 

 xxmaj hypothesis : xxmaj there are people outside .
xxmaj premise : xxmaj two old men in hats doze in the sun outside . 

 xxmaj hypothesis : xxmaj the two men are fishing . 

 xxmaj label : contradiction xxbos xxmaj premise : a man w

In [24]:
learn = language_model_learner(dls_lm, AWD_LSTM, drop_mult=0.3,
                              metrics=[accuracy, Perplexity()]).to_fp16()

In [25]:
learn.fit_one_cycle(1, 2e-2)

epoch,train_loss,valid_loss,accuracy,perplexity,time
0,2.000233,1.841353,0.608383,6.305061,00:48


In [26]:
learn.save('1epoch')

Path('snli_text_files/models/1epoch.pth')

In [27]:
learn.fit_one_cycle(10, 2e-2)

epoch,train_loss,valid_loss,accuracy,perplexity,time
0,1.826160,1.781092,0.615557,5.936337,00:46
1,1.736558,1.682180,0.629567,5.377264,00:46
2,1.633433,1.598004,0.644778,4.943154,00:47
3,1.555565,1.538365,0.654036,4.656970,00:47
4,1.476877,1.490695,0.665352,4.440182,00:47
5,1.426960,1.455481,0.670815,4.286545,00:47
6,1.375850,1.426347,0.678731,4.163463,00:47
7,1.336322,1.409788,0.682671,4.095085,00:47
8,1.319502,1.402401,0.684294,4.064947,00:48
9,1.300478,1.400816,0.684566,4.058512,00:47


In [29]:
TEXT = 'The girl is '
N_WORDS = 40
N_SENTENCES = 2
preds = [learn.predict(TEXT, N_WORDS, temperature=0.75)
        for _ in range(N_SENTENCES)]
preds

['The girl is painting a log . \n\n Hypothesis : The log is on the beach . \n\n Label : entailment Premise : a man , tall black man with glasses and a black shirt , is seated',
 "The girl is smiling while comparing skin 's color . \n\n Hypothesis : The girl is wearing pink . \n\n Label : contradiction Premise : An older man wearing a gray shirt is sitting on a bench"]

In [30]:
learn.fit_one_cycle(20, 2e-2)

epoch,train_loss,valid_loss,accuracy,perplexity,time
0,1.309751,1.399938,0.684862,4.054950,00:47
1,1.320007,1.402589,0.684403,4.065711,00:47
2,1.342217,1.411995,0.682126,4.104136,00:47
3,1.369862,1.422611,0.680039,4.147934,00:47
4,1.375488,1.422242,0.680780,4.146407,00:47
5,1.369254,1.413016,0.683287,4.108327,00:47
6,1.344528,1.403698,0.685765,4.070225,00:47
7,1.333225,1.393059,0.686815,4.027150,00:47
8,1.317224,1.383689,0.689171,3.989590,00:47
9,1.293158,1.369829,0.692295,3.934679,00:47


In [31]:
learn.export('snli.pkl')